# 07 — ETL Encuestas (EVB + ESPC + ISC)

Construye `contexto_encuestas` con las **series temporales clave** de victimización y
percepción de seguridad de las tres encuestas. Tabla LARGA con discriminador `font`.

| font | fuente | territori | qué |
|------|--------|-----------|-----|
| `EVB` | Enquesta Victimització Barcelona 2025 (series `_historico`) | Barcelona | índice victimització + percepció seguretat (1983-2025) |
| `ESPC` | Enquesta Seguretat Pública Catalunya 2022 (índices anuales) | Cataluña | 5 índices (2015-2022) por cistell |
| `ISC` | Indicadors Seguretat Catalunya 2022-2023 | Cataluña | 360 indicadores ya en formato largo |

**Output:** `data/clean/contexto_encuestas.csv`

**Esquema:** `font, anyo, id_territori, territori, nivel_territorial, categoria, subcategoria, indicador, valor, unitat, nota`

Tabla de CONTEXTO: se une por `anyo + territori`. **No mezclar `font`** (metodologías distintas).

⚠️ **CONVENCIONES DE AÑO (importante para joins con datos de crimen):**
- **EVB**: `anyo` = `any_enquesta` (año de la ENCUESTA). La encuesta recoge la victimización
  del periodo previo, así que la edición 2025 usa una etiqueta de año 1 unidad menor que las
  ediciones antiguas (que etiquetaban por año de referencia). Si se cruza con crimen por año
  exacto, considerar un posible rezago de ~1 año. Se preserva el valor original del fichero.
- **ISC**: incluye algún indicador de OBJETIVO futuro (p.ej. año 2030 = objetivo europeo de
  reducción de víctimas de tráfico -50%), no es un dato medido. Está marcado en `indicador`/`nota`.

**Alcance:** las series históricas (lo más útil para análisis temporal). Las tablas cruzadas
por àmbit/districte/sexe de cada edición quedan como extensión futura.

P10: los ficheros EVB tienen tildes en el nombre → se localizan con `os.listdir()`."

In [ ]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

BASE = Path(r'E:\informacion y documentos\Curso analisis de datos IT Academy\Proyecto final\criminalistica_cat')
RAW  = BASE / 'data' / 'raw'
CLEAN = BASE / 'data' / 'clean'
ENC = RAW / 'encuestas'

dim_territorio = pd.read_csv(CLEAN / 'dim_territorio.csv', encoding='utf-8')
ID_CATALUNA = int(dim_territorio[(dim_territorio['fuente']=='ministerio_ine') & (dim_territorio['nivel_territorial']=='ccaa')]['id_territorio'].iloc[0])
print('ID Cataluña CCAA:', ID_CATALUNA)

ESQ = ['font','anyo','id_territori','territori','nivel_territorial','categoria','subcategoria','indicador','valor','unitat','nota']

def buscar(carpeta, substr):
    """Localiza un fichero por substring (nombres con tildes — P10)."""
    cands = [f for f in os.listdir(carpeta) if substr in f and f.endswith('.csv')]
    if not cands:
        raise FileNotFoundError(f'No se encontró "{substr}" en {carpeta}')
    return carpeta / cands[0]

---
## 1. EVB (Barcelona) — series históricas victimización + percepción

In [ ]:
evb_dir = ENC / 'evb' / '2025'

# 1a. Índice de victimización histórico (WIDE: sin/con ciber)
vic = pd.read_csv(buscar(evb_dir, 'indice_victimizacion_historico'), sep=';', encoding='utf-8-sig')
vic_long = vic.melt(
    id_vars=['any_enquesta', 'nota'],
    value_vars=['indice_victimizacion_sin_ciber_pct', 'indice_victimizacion_con_ciber_pct'],
    var_name='ind_raw', value_name='valor'
)
MAP_VIC = {
    'indice_victimizacion_sin_ciber_pct': 'index_victimitzacio_sense_ciber',
    'indice_victimizacion_con_ciber_pct': 'index_victimitzacio_amb_ciber'
}
vic_long['indicador'] = vic_long['ind_raw'].map(MAP_VIC)
vic_long['valor'] = pd.to_numeric(vic_long['valor'], errors='coerce')
vic_long = vic_long.dropna(subset=['valor'])
vic_long = vic_long.rename(columns={'any_enquesta': 'anyo'})
vic_long['unitat'] = 'percentatge'

# 1b. Percepción de seguridad histórica (WIDE: ciutat/barri, escala 0-10)
perc = pd.read_csv(buscar(evb_dir, 'percepcio_seguretat_historic'), sep=';', encoding='utf-8-sig')
perc_long = perc.melt(
    id_vars=['any_enquesta'],
    value_vars=['percepcio_seguretat_ciutat_0a10', 'percepcio_seguretat_barri_0a10'],
    var_name='ind_raw', value_name='valor'
)
MAP_PERC = {
    'percepcio_seguretat_ciutat_0a10': 'percepcio_seguretat_ciutat',
    'percepcio_seguretat_barri_0a10': 'percepcio_seguretat_barri'
}
perc_long['indicador'] = perc_long['ind_raw'].map(MAP_PERC)
perc_long['valor'] = pd.to_numeric(perc_long['valor'], errors='coerce')
perc_long = perc_long.dropna(subset=['valor'])
perc_long = perc_long.rename(columns={'any_enquesta': 'anyo'})
perc_long['nota'] = pd.NA
perc_long['unitat'] = 'escala_0_10'

evb = pd.concat([vic_long, perc_long], ignore_index=True)
evb['font'] = 'EVB'
evb['territori'] = 'Barcelona'
evb['nivel_territorial'] = 'municipi'
evb['id_territori'] = pd.NA
evb['categoria'] = pd.NA
evb['subcategoria'] = pd.NA
evb = evb[ESQ].copy()
print('EVB:', len(evb), 'filas | años', int(evb['anyo'].min()), '-', int(evb['anyo'].max()))
print('  indicadores:', evb['indicador'].unique().tolist())
evb.head(3)

---
## 2. ESPC (Cataluña) — índices anuales por cistell

In [ ]:
espc_raw = pd.read_csv(ENC / 'espc' / '01_indices_anuales_ESPC_2022.csv', sep=',', encoding='utf-8-sig')
print('Cols ESPC:', espc_raw.columns.tolist())
idx_cols = ['index_victimitzacio', 'index_fets_delictius', 'index_multivictimitzacio', 'record_espontani', 'nivell_seguretat_municipi']
espc = espc_raw.melt(
    id_vars=['any', 'cistell'],
    value_vars=idx_cols,
    var_name='indicador', value_name='valor'
)
espc['valor'] = pd.to_numeric(espc['valor'], errors='coerce')
espc = espc.dropna(subset=['valor'])
espc = espc.rename(columns={'any': 'anyo'})
espc['font'] = 'ESPC'
espc['territori'] = 'Cataluña'
espc['nivel_territorial'] = 'ccaa'
espc['id_territori'] = ID_CATALUNA
# cistell = cesta metodológica (año base de referencia) → se guarda como categoria
espc['categoria'] = 'cistell_' + espc['cistell'].astype(int).astype(str)
espc['subcategoria'] = pd.NA
espc['unitat'] = np.where(espc['indicador'].str.startswith('index'), 'percentatge', 'escala')
espc['nota'] = pd.NA
espc = espc[ESQ].copy()
print('ESPC:', len(espc), 'filas | años', sorted(espc['anyo'].unique()))
print('  indicadores:', espc['indicador'].unique().tolist())
print('  cistells:', espc['categoria'].unique().tolist())
espc.head(3)

---
## 3. ISC (Cataluña) — indicadores de seguridad (ya en formato largo)

In [ ]:
isc_raw = pd.read_csv(ENC / 'isc' / 'ISC_2022_2023_dades_seguretat_Catalunya.csv', sep=',', encoding='utf-8-sig')
print('Cols ISC:', isc_raw.columns.tolist())
isc = isc_raw.rename(columns={'any': 'anyo', 'notes': 'nota'}).copy()
isc['anyo'] = pd.to_numeric(isc['anyo'], errors='coerce')
isc = isc.dropna(subset=['anyo'])
isc['anyo'] = isc['anyo'].astype(int)
isc['valor'] = pd.to_numeric(isc['valor'], errors='coerce')
isc = isc.dropna(subset=['valor'])
isc['font'] = 'ISC'
isc['territori'] = 'Cataluña'
isc['nivel_territorial'] = 'ccaa'
isc['id_territori'] = ID_CATALUNA
# (la columna 'font' original del ISC = fuente upstream; se conserva en 'nota')
isc['nota'] = isc['nota'].fillna('').astype(str).str.strip()
isc = isc[ESQ].copy()
print('ISC:', len(isc), 'filas | años', int(isc['anyo'].min()), '-', int(isc['anyo'].max()))
print('  categorias:', isc['categoria'].nunique(), '| indicadores:', isc['indicador'].nunique())
isc.head(3)

---
## 4. Unir y guardar

In [ ]:
contexto_enc = pd.concat([evb, espc, isc], ignore_index=True)
contexto_enc['anyo'] = contexto_enc['anyo'].astype(int)
contexto_enc['id_territori'] = contexto_enc['id_territori'].astype('Int64')

print('contexto_encuestas:', contexto_enc.shape)
print('\nFilas por font:')
print(contexto_enc['font'].value_counts())
print('\nNulos valor:', contexto_enc['valor'].isna().sum())
print('Rango años:', contexto_enc['anyo'].min(), '-', contexto_enc['anyo'].max())
fk = contexto_enc['id_territori'].dropna()
print('id_territori (no nulo) válido en dim_territorio:', fk.isin(dim_territorio['id_territorio']).all(), f'| filas con id: {len(fk)}')
# Cross-check: índice victimización EVB 2010 y 2011 (referencia: 2010 IV~24.8, 2011~24.2)
for a in [2010, 2011]:
    v = contexto_enc[(contexto_enc['font']=='EVB') & (contexto_enc['indicador']=='index_victimitzacio_sense_ciber') & (contexto_enc['anyo']==a)]['valor']
    print(f'  EVB índice victimización {a}:', v.values)

In [ ]:
ruta = CLEAN / 'contexto_encuestas.csv'
contexto_enc.to_csv(ruta, index=False, encoding='utf-8')
print(f'[OK] Guardado {ruta.name}: {len(contexto_enc)} filas')
print('\n¡ETL completo! (notebooks 01-07). Siguiente fase: crear MySQL y cargar data/clean/.')